# Complete indirect prompt-injection base-paper reproduction

This is the single Kaggle orchestration notebook. It clones the GitHub repository and runs the Python implementation from that cloud clone. Select exactly one expensive stage per Kaggle run.


In [1]:
from pathlib import Path
import os
import subprocess

PROJECT_URL = "https://github.com/apusharma45/indirect-pia-detection.git"
PROJECT_DIR = Path("/kaggle/working/indirect-pia-detection")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(f"Refusing to clone into non-empty directory: {PROJECT_DIR}")
else:
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

Cloning into '/kaggle/working/indirect-pia-detection'...


Project directory: /kaggle/working/indirect-pia-detection
7f6a84545f09ed1f9c2b64bfa8a1315302cfc446


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## Capture the untouched Kaggle environment

Run this before any package installation so we can decide whether the author's pinned environment is compatible with the assigned GPU image.

In [2]:
subprocess.run(
    ["python", "scripts/check_environment.py", "--output", "results/environment.txt"],
    check=True,
)

Indirect PIA reproduction environment
generated_at_utc: 2026-10-07T16:06:03.460142+00:00
git_commit: 7f6a84545f09ed1f9c2b64bfa8a1315302cfc446
python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
python_executable: /usr/bin/python3
platform: Linux-6.18.48+-x86_64-with-glibc2.39
kaggle_kernel_run_type: Interactive
cuda_visible_devices: not set

Package versions
----------------
torch: 2.11.0+cu128
transformers: 5.16.1
deepspeed: not installed
accelerate: 1.14.0
datasets: 4.8.5
tokenizers: 0.23.1

PyTorch / CUDA
--------------
import: ok
version: 2.11.0+cu128
cuda_available: True
compiled_cuda_version: 12.8
cudnn_version: 91900
device_count: 2
bf16_supported: True
devices: [{'index': 0, 'name': 'Tesla T4', 'total_memory_bytes': 15636037632, 'total_memory_gib': 14.56, 'compute_capability': '7.5'}, {'index': 1, 'name': 'Tesla T4', 'total_memory_bytes': 15636037632, 'total_memory_gib': 14.56, 'compute_capability': '7.5'}]

nvidia-smi
----------
0, Tesla T4, 15360 MiB, 580.178.04, 7.5
1

CompletedProcess(args=['python', 'scripts/check_environment.py', '--output', 'results/environment.txt'], returncode=0)

## Validate the supplied data

This reads and hashes all JSON datasets and regenerates the two summaries. It does not alter any source data.

In [3]:
subprocess.run(
    ["python", "scripts/inspect_data.py", "--data-dir", "data", "--output-dir", "results"],
    check=True,
)
print((PROJECT_DIR / "results/data_summary.md").read_text(encoding="utf-8"))

Wrote /kaggle/working/indirect-pia-detection/results/data_summary.json
Wrote /kaggle/working/indirect-pia-detection/results/data_summary.md
Status: ok; files: 5; examples: 58848
# Dataset inspection summary

- Status: `ok`
- Data directory: `data`
- Git revision: `7f6a84545f09ed1f9c2b64bfa8a1315302cfc446`
- JSON files: 5
- Total examples: 58848

The script only read and hashed the source datasets; it did not modify them.

## `crafted_instruction_data_alpaca.json`

- Size: 11,460,379 bytes
- SHA-256: `f72822723a36faa3fccc22ff474c61f149533d5899519de153abd4307f986ae3`
- Top level: array
- Examples: 19,157
- Keys: `input`, `instruction`, `output`
- Malformed/non-object records: 0
- Records missing modal fields: 0
- Label fields: `{}`
- Position fields: `{}`
- Attack/category fields: `{}`

Representative records (text is truncated):

```json
[
  {
    "index": 0,
    "record": {
      "input": "4/16",
      "instruction": "Explain why the following fraction is equivalent to 1/4",
      "out

## How execution works

`PROJECT_DIR` is `/kaggle/working/indirect-pia-detection`, created by the first cell. Commands below run with that directory as their working directory, so they invoke the cloned `train_head.py`, `train_classification.py`, `train.py`, `run_detection.py`, `run_purify.py`, and `run_evaluation_instruction.py` files. The notebook only orchestrates and records them; it does not replace their model or attack logic.

Later stages are included now for a complete workflow, but remain gated. Activate one stage only after its prerequisites have succeeded.


In [ ]:
# -------------------------- Stage control --------------------------
# Change only this value between runs. Start with deberta_smoke.
ACTIVE_STAGE = "deberta_smoke"

VALID_STAGES = {
    "audit",
    "deberta_smoke",
    "deberta_full",
    "deberta_eval_squad",
    "deberta_eval_trivia",
    "generative_train",
    "generative_eval_squad",
    "generative_eval_trivia",
    "extraction_train",
    "removal_segmentation_squad",
    "removal_segmentation_trivia",
    "removal_extraction_squad",
    "removal_extraction_trivia",
    "final_asr_segmentation_squad",
    "final_asr_segmentation_trivia",
    "final_asr_extraction_squad",
    "final_asr_extraction_trivia",
}
if ACTIVE_STAGE not in VALID_STAGES:
    raise ValueError(f"Unknown ACTIVE_STAGE: {ACTIVE_STAGE}")

RESULTS_ROOT = PROJECT_DIR / "results"
MODEL_ROOT = PROJECT_DIR / "artifacts/models"

# These defaults are produced by training stages in the current Kaggle session.
# For a later session, attach the saved Kaggle output and replace a path below.
DEBERTA_CHECKPOINT = MODEL_ROOT / "prompt-discriminative-cls-deberta"
GENERATIVE_CHECKPOINT = MODEL_ROOT / "generative-class-llama32-1b"
EXTRACTION_CHECKPOINT = MODEL_ROOT / "ext-llama32-3b"

DEBERTA_MODEL = "microsoft/deberta-v3-base"
GENERATIVE_BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
EXTRACTION_BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
TARGET_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

ATTACKS = ["none", "naive", "ignore", "escape_separation", "completion_real", "completion_realcmb"]
INJECTED_ATTACKS = ATTACKS[1:]
POSITIONS = ["start", "middle", "end"]

print("Active stage:", ACTIVE_STAGE)
print("Python implementation directory:", PROJECT_DIR)


In [ ]:
# Load Hugging Face authentication only when downloading a gated base/target model.
LLAMA_STAGES = {
    "generative_train",
    "extraction_train",
    "final_asr_segmentation_squad",
    "final_asr_segmentation_trivia",
    "final_asr_extraction_squad",
    "final_asr_extraction_trivia",
}
if ACTIVE_STAGE in LLAMA_STAGES:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("This stage requires a Kaggle secret named HF_TOKEN.")
    os.environ["HF_TOKEN"] = hf_token
    del hf_token
    print("HF_TOKEN loaded from Kaggle Secrets (value not displayed).")
else:
    print("This stage does not require a gated Hugging Face model token.")


In [ ]:
import json
import shlex
import sys
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def git_commit():
    return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()

def tail_text(path, count=120):
    path = Path(path)
    if not path.exists():
        return ""
    return "".join(path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-count:])

def run_logged(stage, command, extra_env=None):
    """Run one official entry point and save command, environment, console log, and status."""
    stage_dir = RESULTS_ROOT / stage
    stage_dir.mkdir(parents=True, exist_ok=True)
    for stale_name in ["error.txt", "console.log", "upstream.log"]:
        stale = stage_dir / stale_name
        if stale.exists():
            stale.unlink()

    command_text = shlex.join([str(part) for part in command])
    (stage_dir / "command.txt").write_text(command_text + "\n", encoding="utf-8")
    metadata = {
        "stage": stage,
        "git_commit": git_commit(),
        "started_at_utc": utc_now(),
        "command": command_text,
        "scientific_result": stage not in {"deberta_smoke"},
    }
    (stage_dir / "config.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "scripts/check_environment.py", "--output", str(stage_dir / "environment.txt")],
        cwd=PROJECT_DIR,
        check=True,
        stdout=subprocess.DEVNULL,
    )

    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    print(f"Starting {stage}; full console output -> {stage_dir / 'console.log'}")
    with (stage_dir / "console.log").open("w", encoding="utf-8") as log:
        completed = subprocess.run(command, cwd=PROJECT_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)

    status = {
        "stage": stage,
        "status": "success" if completed.returncode == 0 else "failed",
        "return_code": completed.returncode,
        "finished_at_utc": utc_now(),
    }
    (stage_dir / "status.json").write_text(json.dumps(status, indent=2) + "\n", encoding="utf-8")
    if completed.returncode != 0:
        message = f"Stage {stage} failed with return code {completed.returncode}.\n\n{tail_text(stage_dir / 'console.log')}"
        (stage_dir / "error.txt").write_text(message, encoding="utf-8")
    print(json.dumps(status, indent=2))
    print(tail_text(stage_dir / "console.log"))
    return completed.returncode, stage_dir

def install_dependencies():
    setup_dir = RESULTS_ROOT / "setup"
    setup_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", "configs/kaggle-smoke-requirements.txt"]
    print("Installing the minimal versioned dependencies required by the cloned Python code.")
    with (setup_dir / "dependency_install_log.txt").open("w", encoding="utf-8") as log:
        completed = subprocess.run(command, cwd=PROJECT_DIR, stdout=log, stderr=subprocess.STDOUT)
    if completed.returncode != 0:
        print(tail_text(setup_dir / "dependency_install_log.txt"))
        raise RuntimeError(f"Dependency installation failed with return code {completed.returncode}")
    print(tail_text(setup_dir / "dependency_install_log.txt", 30))

def require_checkpoint(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} checkpoint not found at {path}. Attach the prior Kaggle output and update the Stage control path.")

def parse_log(mode, stage_dir, dataset_name, sample_count=900):
    output_stem = {"detection": "detection_results", "removal": "removal_results", "asr": "asr_results"}[mode]
    command = [
        sys.executable, "scripts/parse_evaluation_log.py",
        "--mode", mode,
        "--log", str(stage_dir / "upstream.log"),
        "--dataset", dataset_name,
        "--sample-count", str(sample_count),
        "--csv-output", str(stage_dir / f"{output_stem}.csv"),
        "--json-output", str(stage_dir / f"{output_stem}.json"),
    ]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

def deepspeed_prefix(port):
    return [sys.executable, "-m", "deepspeed.launcher.runner", "--include", "localhost:0", "--master_port", str(port)]


In [ ]:
# The smoke runner performs its own captured install. Other compute stages install the same minimum set here.
if ACTIVE_STAGE not in {"audit", "deberta_smoke"}:
    install_dependencies()
else:
    print("Dependency setup is handled by this stage or is not needed.")


## Execute the selected stage

Only the branch matching `ACTIVE_STAGE` runs. Training checkpoints go under `artifacts/models/` and are not committed to Git. Evaluation stages call the official scripts and then parse their text logs without changing their evaluation logic.


In [ ]:
DATASETS = {
    "squad": ("data/crafted_instruction_data_squad_injection_qa.json", "Inj-SQuAD"),
    "trivia": ("data/crafted_instruction_data_tri_injection_qa.json", "Inj-TriviaQA"),
}

last_stage_dir = None
last_return_code = 0

if ACTIVE_STAGE == "audit":
    print("Audit and data validation completed in the earlier cells; no model command selected.")

elif ACTIVE_STAGE == "deberta_smoke":
    smoke = subprocess.run(["bash", "scripts/run_deberta_smoke.sh", "--install-deps"], cwd=PROJECT_DIR)
    last_return_code = smoke.returncode
    last_stage_dir = RESULTS_ROOT / "deberta_smoke"

elif ACTIVE_STAGE == "deberta_full":
    if DEBERTA_CHECKPOINT.exists():
        raise FileExistsError(f"Refusing to overwrite existing checkpoint: {DEBERTA_CHECKPOINT}")
    stage_dir = RESULTS_ROOT / ACTIVE_STAGE
    stage_dir.mkdir(parents=True, exist_ok=True)
    command = deepspeed_prefix(1113) + [
        "train_head.py",
        "--model_name_or_path", DEBERTA_MODEL,
        "--instruction_train_data_path", "data/crafted_instruction_data_alpaca.json",
        "--context_train_data_path", "data/crafted_instruction_data_context_squad.json",
        "--eval_data_path", "data/crafted_instruction_data_squad_injection_qa.json",
        "--bf16",
        "--save_path", str(DEBERTA_CHECKPOINT),
        "--max_epochs", "1",
        "--train_batch_size", "4",
        "--micro_train_batch_size", "4",
        "--learning_rate", "1e-5",
        "--l2", "0",
        "--lr_scheduler", "cosine",
        "--logging_steps", "1",
        "--adam_offload",
        "--zero_stage", "1",
        "--seed", "42",
        "--inject_rate", "0.6",
        "--log_file", str(stage_dir / "upstream.log"),
    ]
    last_return_code, last_stage_dir = run_logged(ACTIVE_STAGE, command, {"DS_SKIP_CUDA_CHECK": "1"})

elif ACTIVE_STAGE.startswith("deberta_eval_") or ACTIVE_STAGE.startswith("generative_eval_"):
    family = "deberta" if ACTIVE_STAGE.startswith("deberta") else "generative"
    domain = "squad" if ACTIVE_STAGE.endswith("squad") else "trivia"
    data_path, dataset_name = DATASETS[domain]
    checkpoint = DEBERTA_CHECKPOINT if family == "deberta" else GENERATIVE_CHECKPOINT
    require_checkpoint(checkpoint, family)
    stage_dir = RESULTS_ROOT / ACTIVE_STAGE
    stage_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "run_detection.py",
        "--model_path", str(checkpoint),
        "--user_data_path", data_path,
        # Parsed by the baseline but not loaded; use an existing file instead of the missing README path.
        "--injected_instruction_data_path", "data/crafted_instruction_data_alpaca.json",
        "--attack", *ATTACKS,
        "--sides", *POSITIONS,
        "--seed", "42",
        "--log_path", str(stage_dir / "upstream.log"),
    ]
    last_return_code, last_stage_dir = run_logged(ACTIVE_STAGE, command)
    if last_return_code == 0:
        parse_log("detection", last_stage_dir, dataset_name)

elif ACTIVE_STAGE == "generative_train":
    if GENERATIVE_CHECKPOINT.exists():
        raise FileExistsError(f"Refusing to overwrite existing checkpoint: {GENERATIVE_CHECKPOINT}")
    stage_dir = RESULTS_ROOT / ACTIVE_STAGE
    stage_dir.mkdir(parents=True, exist_ok=True)
    command = deepspeed_prefix(1114) + [
        "train_classification.py",
        "--model_name_or_path", GENERATIVE_BASE_MODEL,
        "--instruction_train_data_path", "data/crafted_instruction_data_alpaca.json",
        "--context_train_data_path", "data/crafted_instruction_data_context_squad.json",
        "--eval_data_path", "data/crafted_instruction_data_squad_injection_qa.json",
        "--bf16",
        "--save_path", str(GENERATIVE_CHECKPOINT),
        "--max_epochs", "1",
        "--train_batch_size", "4",
        "--micro_train_batch_size", "4",
        "--learning_rate", "1e-5",
        "--l2", "0",
        "--lr_scheduler", "cosine",
        "--logging_steps", "1",
        "--adam_offload",
        "--zero_stage", "1",
        "--seed", "42",
        "--inject_rate", "0.6",
        "--log_file", str(stage_dir / "upstream.log"),
    ]
    last_return_code, last_stage_dir = run_logged(ACTIVE_STAGE, command, {"DS_SKIP_CUDA_CHECK": "1"})

elif ACTIVE_STAGE == "extraction_train":
    if EXTRACTION_CHECKPOINT.exists():
        raise FileExistsError(f"Refusing to overwrite existing checkpoint: {EXTRACTION_CHECKPOINT}")
    stage_dir = RESULTS_ROOT / ACTIVE_STAGE
    stage_dir.mkdir(parents=True, exist_ok=True)
    command = deepspeed_prefix(1145) + [
        "train.py",
        "--model_name_or_path", EXTRACTION_BASE_MODEL,
        "--instruction_train_data_path", "data/crafted_instruction_data_alpaca.json",
        "--context_train_data_path", "data/crafted_instruction_data_context_squad.json",
        "--eval_data_path", "data/crafted_instruction_data_squad_injection_qa.json",
        "--bf16",
        "--save_path", str(EXTRACTION_CHECKPOINT),
        "--max_epochs", "1",
        "--train_batch_size", "4",
        "--micro_train_batch_size", "4",
        "--learning_rate", "1e-5",
        "--l2", "0",
        "--lr_scheduler", "cosine",
        "--logging_steps", "1",
        "--adam_offload",
        "--zero_stage", "1",
        "--seed", "42",
        "--log_file", str(stage_dir / "upstream.log"),
    ]
    last_return_code, last_stage_dir = run_logged(ACTIVE_STAGE, command, {"DS_SKIP_CUDA_CHECK": "1"})

elif ACTIVE_STAGE.startswith("removal_"):
    method = "cls" if "segmentation" in ACTIVE_STAGE else "ext"
    domain = "squad" if ACTIVE_STAGE.endswith("squad") else "trivia"
    data_path, dataset_name = DATASETS[domain]
    require_checkpoint(DEBERTA_CHECKPOINT, "DeBERTa")
    if method == "ext":
        require_checkpoint(EXTRACTION_CHECKPOINT, "extraction")
    stage_dir = RESULTS_ROOT / ACTIVE_STAGE
    stage_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "run_purify.py",
        "--model_path", str(DEBERTA_CHECKPOINT),
        "--user_data_path", data_path,
        "--injected_instruction_data_path", "data/crafted_instruction_data_alpaca.json",
        "--attack", *INJECTED_ATTACKS,
        "--sides", *POSITIONS,
        "--purify_method", method,
        "--seed", "42",
        "--log_path", str(stage_dir / "upstream.log"),
    ]
    if method == "ext":
        command[4:4] = ["--ext_model_path", str(EXTRACTION_CHECKPOINT)]
    last_return_code, last_stage_dir = run_logged(ACTIVE_STAGE, command)
    if last_return_code == 0:
        parse_log("removal", last_stage_dir, dataset_name)

elif ACTIVE_STAGE.startswith("final_asr_"):
    method = "cls" if "segmentation" in ACTIVE_STAGE else "ext"
    domain = "squad" if ACTIVE_STAGE.endswith("squad") else "trivia"
    data_path, dataset_name = DATASETS[domain]
    require_checkpoint(DEBERTA_CHECKPOINT, "DeBERTa")
    require_checkpoint(EXTRACTION_CHECKPOINT, "extraction")
    stage_dir = RESULTS_ROOT / ACTIVE_STAGE
    stage_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "run_evaluation_instruction.py",
        "--model_path", TARGET_MODEL,
        "--filter_bot", str(DEBERTA_CHECKPOINT),
        "--extract_bot", str(EXTRACTION_CHECKPOINT),
        "--data_path", data_path,
        "--attack", *INJECTED_ATTACKS,
        "--defense", "filter",
        "--injection_type", "adv",
        "--purify_method", method,
        "--side", *POSITIONS,
        "--seed", "42",
        "--log_path", str(stage_dir / "upstream.log"),
    ]
    last_return_code, last_stage_dir = run_logged(ACTIVE_STAGE, command)
    if last_return_code == 0:
        parse_log("asr", last_stage_dir, dataset_name)

else:
    raise AssertionError(f"Stage dispatch missing for {ACTIVE_STAGE}")

print("Selected stage finished with return code:", last_return_code)


## Review compact evidence

The full logs remain in the cloned repository. This cell prints structured files and only the end of large logs so the saved notebook remains readable.


In [ ]:
if last_stage_dir is not None:
    print("Result directory:", last_stage_dir)
    for name in [
        "status.json", "metrics.json", "config.json", "error.txt",
        "detection_results.json", "removal_results.json", "asr_results.json",
        "environment.txt", "command.txt",
    ]:
        result_path = last_stage_dir / name
        if result_path.exists():
            print(f"\n===== {name} =====")
            print(result_path.read_text(encoding="utf-8", errors="replace"))
    for name in ["dependency_install_log.txt", "training_log.txt", "console.log", "upstream.log"]:
        log_path = last_stage_dir / name
        if log_path.exists():
            print(f"\n===== final 120 lines: {name} =====")
            print(tail_text(log_path, 120))
else:
    print("No compute-stage result directory was created.")


## Save and continue

After the selected stage finishes:

1. Use Kaggle **Save Version** and keep the notebook outputs.
2. For training stages, enable saving output files so `artifacts/models/` can be attached to a later session; checkpoints are intentionally not pushed to ordinary Git.
3. Update/export this same `indirect-pia-detection.ipynb` to GitHub.
4. Pull GitHub locally and let Codex inspect the structured output printed above.

Do not activate the next stage until the current result has been validated. The intended order is documented in `docs/kaggle_notebook_guide.md`.
